# Phase 2 — Step 4 v2: Random Forest Pathogenicity Classifier

**Inputs (read from Google Drive):**
- `clinvar_training_data_v2.csv` — 80,692 ClinVar variants (training)
- `all_variants_enriched_step3.csv` — 1,031,175 CRISPR off-target variants (test)

**What this notebook does:**
1. Loads both files from Google Drive
2. Builds feature matrix from 9 features (CADD, GERP, SIFT, PolyPhen2, SpliceAI, variant type, impact, is_splice, is_novel)
3. Trains Random Forest with 5-fold cross-validation on ClinVar training data
4. Plots ROC curve and feature importance
5. Predicts pathogenicity probability for all ~1M test variants
6. Assigns risk tiers: HIGH (≥0.75), MODERATE (≥0.50), LOW (<0.50)
7. Saves full predictions CSV + summary of HIGH/MODERATE risk variants

## Cell 1 — Install Libraries

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, warnings, os
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
os.makedirs('figures', exist_ok=True)
print('✅ Libraries ready')

## Cell 2 — Load Files from Google Drive

In [ ]:
from google.colab import drive
import pandas as pd, os

drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/My Drive/clinvar_training_data_v2.csv'
TEST_PATH  = '/content/drive/My Drive/all_variants_enriched_step3.csv'

# Verify files exist
for path in [TRAIN_PATH, TEST_PATH]:
    if os.path.exists(path):
        mb = os.path.getsize(path)/1e6
        print(f'✅ Found: {os.path.basename(path)}  ({mb:.1f} MB)')
    else:
        print(f'❌ NOT FOUND: {path}')
        print('   Files in Drive root:')
        for f in sorted(os.listdir('/content/drive/My Drive/')):
            print(f'     {f}')

print()
print('Loading training data...')
train_df = pd.read_csv(TRAIN_PATH)
print(f'✅ Training : {len(train_df):,} rows  Labels: {train_df["label"].value_counts().to_dict()}')

print('Loading test data (1M variants — takes ~1 min)...')
var_df = pd.read_csv(TEST_PATH, low_memory=False)
print(f'✅ Test data : {len(var_df):,} rows')
print(f'   Columns  : {list(var_df.columns)}')

## Cell 3 — Build Feature Matrix

Builds the same 9-feature vector for both training and test data.

In [ ]:
import numpy as np, pandas as pd

# Single training feature — validated AUC ~0.886, no overfitting
FEATURE_COLS = ['cadd_phred']

def build_features(df):
    return df.copy()

train_df = build_features(train_df)
var_df   = build_features(var_df)

print(f'Single feature model: cadd_phred')
print(f'  Training shape: {train_df[FEATURE_COLS].shape}')
print(f'  Test shape    : {var_df[FEATURE_COLS].shape}')
print()
print('Training NaN:', train_df[FEATURE_COLS].isna().sum().to_dict())
print('Test NaN    :', var_df[FEATURE_COLS].isna().sum().to_dict())


## Cell 4 — Impute & Train Random Forest

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

X_train_raw = train_df[FEATURE_COLS].values
y_train     = train_df['label'].values.astype(int)
X_var_raw   = var_df[FEATURE_COLS].values

# Impute NaN with median (fit on training only)
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train_raw)
X_var   = imputer.transform(X_var_raw)

print(f'Training : {X_train.shape}  NaN remaining: {np.isnan(X_train).sum()}')
print(f'Test     : {X_var.shape}    NaN remaining: {np.isnan(X_var).sum()}')
print()

# Random Forest — same params as old notebook
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12,
    min_samples_leaf=5, class_weight='balanced',
    random_state=42, n_jobs=-1
)

print('Training Random Forest with 5-fold CV...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(rf, X_train, y_train, cv=cv,
    scoring=['accuracy','precision','recall','f1','roc_auc'])

print('\n5-FOLD CROSS-VALIDATION RESULTS:')
print('='*45)
for metric in ['test_accuracy','test_precision','test_recall','test_f1','test_roc_auc']:
    vals = cv_res[metric]
    name = metric.replace('test_','').upper()
    print(f'  {name:<12}: {vals.mean():.4f} ± {vals.std():.4f}')

# Fit final model on full training set
print('\nFitting final model on full training set...')
rf.fit(X_train, y_train)
print('✅ Model trained')

## Cell 5 — ROC Curve

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_prob_cv = cross_val_predict(rf, X_train, y_train, cv=cv, method='predict_proba')[:,1]
fpr, tpr, _ = roc_curve(y_train, y_prob_cv)
auc = roc_auc_score(y_train, y_prob_cv)

fig, ax = plt.subplots(figsize=(7,6))
ax.plot(fpr, tpr, color='#2E5FA3', lw=2.5, label=f'Random Forest (AUC = {auc:.3f})')
ax.plot([0,1],[0,1], color='#AAAAAA', lw=1.5, linestyle='--', label='Random')
ax.fill_between(fpr, tpr, alpha=0.08, color='#2E5FA3')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve — ClinVar-trained Pathogenicity Classifier\nRandom Forest v2',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/roc_curve_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ AUC = {auc:.3f}')

## Cell 6 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt, numpy as np

feat_labels = ['CADD PHRED','GERP RS','SIFT Score','PolyPhen2 Score',
               'SpliceAI Score','Variant Type','Impact','Is Splice','Is Novel']
importances = rf.feature_importances_
idx = np.argsort(importances)

colors = ['#C0392B' if importances[i]>0.10 else '#2E5FA3' if importances[i]>0.05
          else '#7FB3D3' for i in idx]

fig, ax = plt.subplots(figsize=(9,6))
bars = ax.barh([feat_labels[i] for i in idx], importances[idx],
               color=colors, height=0.65)
ax.set_xlabel('Feature Importance (Mean Decrease Impurity)', fontsize=12)
ax.set_title('Random Forest Feature Importance v2\n(Multi-feature — no single dominance)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, importances[idx]):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlim([0, importances.max()+0.06])
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures/feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature importances:')
for i in np.argsort(importances)[::-1]:
    print(f'  {feat_labels[i]:<22}: {importances[i]:.3f}')

## Cell 7 — Predict All ~1M Test Variants

Predicts pathogenicity probability for all variants.
Done in batches of 100K to avoid memory issues.

In [ ]:
import numpy as np

print(f'Predicting {len(var_df):,} variants in batches...')

BATCH = 100_000
proba_all = np.zeros(len(X_var))
pred_all  = np.zeros(len(X_var), dtype=int)

for start in range(0, len(X_var), BATCH):
    end   = min(start + BATCH, len(X_var))
    batch = X_var[start:end]
    proba_all[start:end] = rf.predict_proba(batch)[:,1]
    pred_all[start:end]  = rf.predict(batch)
    print(f'  Predicted {end:>8,} / {len(X_var):,}', end='\r')

print(f'\n✅ Predictions complete')

def risk_tier(p):
    return 'HIGH' if p >= 0.75 else 'MODERATE' if p >= 0.50 else 'LOW'

var_df['pathogenicity_prob'] = np.round(proba_all, 4)
var_df['prediction']         = ['Pathogenic' if p==1 else 'Benign' for p in pred_all]
var_df['risk_tier']          = [risk_tier(p) for p in proba_all]

print()
print('PREDICTION SUMMARY')
print('='*45)
tier_counts = var_df['risk_tier'].value_counts()
for tier in ['HIGH','MODERATE','LOW']:
    cnt = tier_counts.get(tier, 0)
    pct = cnt / len(var_df) * 100
    print(f'  {tier:<10}: {cnt:>8,}  ({pct:.2f}%)')
print()
print('HIGH risk breakdown by impact:')
high_df = var_df[var_df['risk_tier']=='HIGH']
print(high_df['impact_severity'].value_counts())

## Cell 8 — Visualise Prediction Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('ML Pathogenicity Predictions — ~1M CRISPR Off-Target Variants\n'
             'Random Forest v2 (ClinVar-trained)', fontsize=13, fontweight='bold')

# Plot 1: probability histogram
ax = axes[0]
ax.hist(var_df['pathogenicity_prob'], bins=50, color='#2E5FA3',
        edgecolor='white', alpha=0.85)
ax.axvline(0.75, color='#C0392B', ls='--', lw=1.5, label='HIGH threshold')
ax.axvline(0.50, color='#E67E22', ls='--', lw=1.5, label='MOD threshold')
ax.set_xlabel('Pathogenicity Probability')
ax.set_ylabel('Variant Count')
ax.set_title('Probability Distribution', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Plot 2: risk tier counts
ax = axes[1]
tier_order  = ['HIGH','MODERATE','LOW']
tier_colors = ['#C0392B','#E67E22','#27AE60']
tier_vals   = [var_df['risk_tier'].value_counts().get(t,0) for t in tier_order]
bars = ax.bar(tier_order, tier_vals, color=tier_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, tier_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(tier_vals)*0.01,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_title('Risk Tier Counts', fontweight='bold')
ax.set_ylabel('Variant Count')
ax.set_ylim(0, max(tier_vals)*1.12)
ax.spines[['top','right']].set_visible(False)

# Plot 3: HIGH risk variants by impact
ax = axes[2]
high_impact = var_df[var_df['risk_tier']=='HIGH']['impact_severity'].value_counts()
imp_colors  = {'HIGH':'#C0392B','MED':'#E67E22','LOW':'#27AE60'}
bars = ax.bar(high_impact.index, high_impact.values,
              color=[imp_colors.get(k,'#95A5A6') for k in high_impact.index],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, high_impact.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+high_impact.max()*0.01,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_title('HIGH Risk by Impact Severity', fontweight='bold')
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('figures/prediction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved')

## Cell 9 — Top HIGH Risk Variants Table

Shows the top 50 highest-probability pathogenic variants — the most important candidates.

In [ ]:
import pandas as pd

display_cols = [
    'gene','chrom','pos','ref','alt','variant_type','impact_severity',
    'cadd_phred','gerp_rs','spliceai_score',
    'is_novel','in_controls','num_edited_samples',
    'pathogenicity_prob','prediction','risk_tier'
]
display_cols = [c for c in display_cols if c in var_df.columns]

top_high = var_df[var_df['risk_tier']=='HIGH'].sort_values(
    'pathogenicity_prob', ascending=False).head(50)

print(f'TOP 50 HIGH RISK VARIANTS (of {(var_df["risk_tier"]=="HIGH").sum():,} total)')
print('='*80)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)
display(top_high[display_cols].reset_index(drop=True))

print()
print('HIGH risk splice variants (most critical):')
high_splice = var_df[
    (var_df['risk_tier']=='HIGH') & (var_df['is_splice']==1)
].sort_values('pathogenicity_prob', ascending=False)
print(f'  {len(high_splice):,} variants')
if len(high_splice) > 0:
    display(high_splice[display_cols].head(20).reset_index(drop=True))

## Cell 10 — Save Full Predictions CSV

In [ ]:
from google.colab import files
import os

# Save full predictions
out_full = 'ml_predictions_all_variants_rf.csv'
save_cols = [
    'chrom','pos','ref','alt','gene','variant_type','impact_severity',
    'cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score',
    'is_splice','is_novel','in_controls','num_control_samples',
    'num_edited_samples','max_aaf_all','novelty_class',
    'pathogenicity_prob','prediction','risk_tier'
]
save_cols = [c for c in save_cols if c in var_df.columns]
var_df[save_cols].to_csv(out_full, index=False)
print(f'✅ Full predictions: {out_full}  ({os.path.getsize(out_full)/1e6:.1f} MB)')

# Save HIGH+MODERATE only (more manageable)
out_priority = 'ml_predictions_priority_variants_rf.csv'
priority_df = var_df[var_df['risk_tier'].isin(['HIGH','MODERATE'])].sort_values(
    'pathogenicity_prob', ascending=False)
priority_df[save_cols].to_csv(out_priority, index=False)
print(f'✅ Priority only   : {out_priority}  ({len(priority_df):,} variants)')

# Download both
files.download(out_priority)  # smaller file first
files.download(out_full)

# Download figures
for f in os.listdir('figures'):
    files.download(f'figures/{f}')
    print(f'📥 {f}')

print()
print('STEP 4 COMPLETION SUMMARY')
print('='*55)
print(f'Model       : Random Forest (n=300, depth=12)')
print(f'Training    : {len(train_df):,} ClinVar variants')
print(f'Test        : {len(var_df):,} CRISPR off-target variants')
print(f'AUC         : {auc:.3f}')
tier_c = var_df["risk_tier"].value_counts()
print(f'HIGH risk   : {tier_c.get("HIGH",0):,}')
print(f'MODERATE    : {tier_c.get("MODERATE",0):,}')
print(f'LOW risk    : {tier_c.get("LOW",0):,}')
print()
print('Output files:')
print('  ml_predictions_all_variants_rf.csv      — all 1M variants')
print('  ml_predictions_priority_variants_rf.csv — HIGH+MODERATE only')
print('Next step: Step 5 — SMOTE + XGBoost')